In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime

In [0]:
tables = spark.catalog.listTables("platform_monitoring")

excluded_tables = [
    "table_metadata",
    "table_metadata_history",
    "access_logs",
    "asset_health",
    "job_history",
    "job_health",
    "governance_violations",
    "alerts",
    "data_quality_metrics",
    "storage_metrics"
]

business_tables = [
    t.name
    for t in tables
    if t.name not in excluded_tables
]

In [0]:
storage_rows = []

for table in business_tables:

    df = spark.table(f"platform_monitoring.{table}")

    row_count = df.count()

    # Simple estimation:
    # Assume each row averages 500 bytes
    estimated_size_mb = round((row_count * 500) / (1024 * 1024), 2)

    # Example storage cost:
    # $0.023 per GB per month
    estimated_size_gb = estimated_size_mb / 1024
    monthly_cost = round(estimated_size_gb * 0.023, 4)

    if estimated_size_mb > 150:
        status = "HIGH"

    elif estimated_size_mb > 50:
        status = "MEDIUM"

    else:
        status = "LOW"

    storage_rows.append(
        Row(
            table_name=table,
            row_count=row_count,
            estimated_size_mb=estimated_size_mb,
            estimated_monthly_cost_usd=monthly_cost,
            storage_status=status,
            collection_time=datetime.now()
        )
    )

In [0]:
storage_rows = []

for table in business_tables:

    df = spark.table(f"platform_monitoring.{table}")

    row_count = df.count()

    # Simple estimation:
    # Assume each row averages 500 bytes
    estimated_size_mb = round((row_count * 500) / (1024 * 1024), 2)

    # Example storage cost:
    # $0.023 per GB per month
    estimated_size_gb = estimated_size_mb / 1024
    monthly_cost = round(estimated_size_gb * 0.023, 4)

    if estimated_size_mb > 150:
        status = "HIGH"

    elif estimated_size_mb > 50:
        status = "MEDIUM"

    else:
        status = "LOW"

    storage_rows.append(
        Row(
            table_name=table,
            row_count=row_count,
            estimated_size_mb=estimated_size_mb,
            estimated_monthly_cost_usd=monthly_cost,
            storage_status=status,
            collection_time=datetime.now()
        )
    )

In [0]:
storage_df = spark.createDataFrame(storage_rows)

In [0]:
storage_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
    "platform_monitoring.storage_metrics"
)

In [0]:
spark.table(
    "platform_monitoring.storage_metrics"
).orderBy(
    F.desc("estimated_size_mb")
).show(50, False)

+------------+---------+-----------------+--------------------------+--------------+--------------------------+
|table_name  |row_count|estimated_size_mb|estimated_monthly_cost_usd|storage_status|collection_time           |
+------------+---------+-----------------+--------------------------+--------------+--------------------------+
|orders      |500000   |238.42           |0.0054                    |HIGH          |2026-06-30 17:39:47.334937|
|sales       |250000   |119.21           |0.0027                    |MEDIUM        |2026-06-30 17:39:49.810931|
|transactions|200000   |95.37            |0.0021                    |MEDIUM        |2026-06-30 17:39:51.328181|
|payments    |200000   |95.37            |0.0021                    |MEDIUM        |2026-06-30 17:39:47.767767|
|customers   |100000   |47.68            |0.0011                    |LOW           |2026-06-30 17:39:46.325395|
|sessions    |100000   |47.68            |0.0011                    |LOW           |2026-06-30 17:39:50.